In [124]:
import os, sys
from dotenv import load_dotenv
load_dotenv(".env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

LLM_MODEL = os.getenv("LLM_MODEL", "gemini/gemini-flash-lite-latest")
_HAS_KEY = bool(os.getenv("GEMINI_API_KEY") or os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"))
PYTHON_BIN = sys.executable  # used to launch MCP servers, only relevant when USE_MCP is True

print("LLM model:", LLM_MODEL, "| API key set:", _HAS_KEY)


LLM model: gemini/gemini-flash-lite-latest | API key set: True


In [2]:
# Readiness check: confirms the libraries import cleanly
import litellm

status = {}
for name, module in [("litellm", "litellm"), ("langchain-core", "langchain_core"),
                     ("langchain-litellm", "langchain_litellm"), ("mcp (client SDK)", "mcp"),
                     ("fastmcp (server framework)", "fastmcp")]:
    try:
        __import__(module)
        status[name] = "ok"
    except Exception as e:
        status[name] = f"import failed: {e}"
status["API key"] = "set" if _HAS_KEY else "NOT set  <-- offline fallbacks will run instead"

print("Environment readiness")
print("---------------------")
for k, v in status.items():
    print(f"  {k:>28} : {v}")


c:\Users\bharath-asus\Documents\LEARNING\Documentation\Articles\agentic_ai_projects\wealth_pilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment readiness
---------------------
                       litellm : ok
                langchain-core : ok
             langchain-litellm : ok
              mcp (client SDK) : ok
    fastmcp (server framework) : ok
                       API key : set


### The MCP client helper

In [3]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def call_mcp_tool(command: str, args: list[str], tool_name: str, arguments: dict, timeout: float = 20.0):
    '''Synchronous wrapper around one MCP tool call.

    command, args: how to start the server (e.g. python, ["-m", "some_server_module"]).
    tool_name, arguments: which tool to call on that server, and with what arguments.
    Returns the tool's text result, or raises if the server reports an error or times out.
    '''
    async def _call():
        server_params = StdioServerParameters(command=command, args=args)
        with open(os.devnull, "w") as devnull:
            async with stdio_client(server_params, errlog=devnull) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    result = await session.call_tool(tool_name, arguments)
                    if result.isError:
                        raise RuntimeError(str(result.content[0].text if result.content else result))
                    return result.content[0].text if result.content else None
    return asyncio.run(asyncio.wait_for(_call(), timeout=timeout))


def _unwrap_exception_group(exc):
    """anyio's TaskGroup (used internally by both stdio_client and ClientSession) can wrap
    a failure in more than one layer of ExceptionGroup. Keep unwrapping .exceptions[0] until
    we reach something that isn't a group -- that's the actual root cause. Without this, a
    fallback message would show a generic 'ExceptionGroup' instead of the real underlying
    error, making it much harder to tell what actually went wrong."""
    while hasattr(exc, "exceptions") and exc.exceptions:
        exc = exc.exceptions[0]
    return exc

print("call_mcp_tool and _unwrap_exception_group defined.")


call_mcp_tool and _unwrap_exception_group defined.


---
# Risk Analysis Agent with Tools

In [58]:
import sqlite3
import pandas as pd

## create a SQLite database and store the CSVs data into it

applicants_df = pd.read_csv("../data/wealthpilot/mock_api/applicants.csv")
loan_applications_df = pd.read_csv("../data/wealthpilot/mock_api/loan_applications.csv")
underwriting_reference_df = pd.read_csv("../data/wealthpilot/mock_api/underwriting_reference.csv")
bureau_df = pd.read_csv("../data/wealthpilot/mock_api/bureau_reports.csv")
bank_statement_df = pd.read_csv("../data/wealthpilot/mock_api/bank_statements.csv")
ref_rate_df = pd.read_csv("../data/wealthpilot/mock_api/reference_rates.csv")
risk_grades_df = pd.read_csv("../data/wealthpilot/mock_api/risk_grades.csv")
sector_policies_df = pd.read_csv("../data/wealthpilot/mock_api/sector_policies.csv")

conn = sqlite3.connect("wealthpilot.db")

def create_table(df: pd.DataFrame, table_name: str):
    df.to_sql(table_name, conn, if_exists="replace", index=False)

create_table(bureau_df, "bureau_table")
create_table(bank_statement_df, "bank_statement_table")
create_table(applicants_df, "applicants_table")
create_table(loan_applications_df, "loan_applications_table")
create_table(underwriting_reference_df, "underwriting_reference_table")
create_table(ref_rate_df, "ref_rate_table")
create_table(risk_grades_df, "risk_grades_table")
create_table(sector_policies_df, "sector_policies_table")

conn.close()

### DSCR Calculator

$$
\text{DSCR} =  \frac{\text{ebitda\_inr}}{\text{existing\_annual\_debt\_service\_inr + proposed\_annual\_debt\_service\_inr}}
$$


In [87]:
### The debt-service-coverage calculator tool

def calculate_dscr(applicant_id: str) -> dict:
    """
    Calculate Debt Service Coverage Ratio (DSCR) for a given applicant ID
    DSCR = ebitda / (existing_annual_debt_service_inr + proposed_annual_debt_service_inr).
    Args:
        applicant_id: The ID of the applicant for whom to calculate DSCR.
    """

    conn = sqlite3.connect("wealthpilot.db")
    conn.row_factory = sqlite3.Row

    try:
        row = conn.execute(
        """
        SELECT app.ebitda_inr as ebitda_inr,
        loan.requested_amount_inr as requested_inr,
        loan.requested_tenor_months as tenor_months,
        loan.existing_annual_debt_service_inr as existing_annual_debt_service_inr,
        app.sector as sector
        FROM applicants_table app 
        LEFT JOIN loan_applications_table loan on app.applicant_id = loan.applicant_id
        WHERE app.applicant_id = ?
        """, (applicant_id,)
        ).fetchone()
    except Exception as e:
        conn.close()
        return {
            "error": f"Database query failed: {e}"
        }

    r = 14.0 / 1200
    emi = row['requested_inr'] * r / (1 - (1 + r) ** -row['tenor_months'])
    proposed =  round(emi * min(12, row['tenor_months']))
        
    if row['existing_annual_debt_service_inr'] is not None and proposed is not None:
        dscr = row['ebitda_inr'] / (row['existing_annual_debt_service_inr'] + proposed)

    return {
        #"proposed_annual_debt_service_inr" : proposed,
        "dscr": round(dscr, 2),
        "sector": row['sector']
    }

In [88]:
# self-check: calculate DSCR for a sample applicant ID
calculate_dscr('ASH-A-00000')

{'dscr': 1.93, 'sector': 'Light Engineering'}

### Sector Policy's DSCR

In [63]:
def get_sector_dscr(sector_name: str) -> dict:
    """Retrieve the sector DSCR."""

    conn = sqlite3.connect("wealthpilot.db")
    conn.row_factory = sqlite3.Row

    row = conn.execute(
        """
        SELECT sector, dscr_floor as sector_dscr_floor
        FROM sector_policies_table
        WHERE sector = ?
        """,
        (sector_name,)
    ).fetchone()

    conn.close()
    if row is None:
        return {"error": "Sector DSCR details not found"}
    return dict(row)

get_sector_dscr('Textiles')

{'sector': 'Textiles', 'sector_dscr_floor': 1.3}

### Credit-bureau API

In [82]:
def get_credit_profile(applicant_id: str) -> dict:
    """Retrieve credit bureau information for a applicant."""

    conn = sqlite3.connect("wealthpilot.db")
    conn.row_factory = sqlite3.Row

    row = conn.execute(
        """
        SELECT bureau_score
        FROM bureau_table
        WHERE applicant_id = ?
        """,
        (applicant_id,)
    ).fetchone()

    conn.close()

    if row is None:
        return {"error": "Applicant details not found"}

    return dict(row)

In [83]:
# self-check
get_credit_profile('ASH-A-00000')

{'bureau_score': 374}

### Bank-statement API

In [ ]:
def get_bank_statement(applicant_id: str) -> dict:
    """Retrieve bank statement information for a applicant."""

    conn = sqlite3.connect("wealthpilot.db")
    conn.row_factory = sqlite3.Row

    row = conn.execute(
        """
        SELECT *
        FROM bank_statement_table
        WHERE applicant_id = ?
        """,
        (applicant_id,)
    ).fetchone()

    conn.close()

    if row is None:
        return {"error": "Applicant details not found"}

    return dict(row)

In [20]:
get_bank_statement('ASH-A-00000')

{'statement_id': 'ASH-S-000128',
 'applicant_id': 'ASH-A-00000',
 'month': '2025-05',
 'inflow_inr': 7140941,
 'outflow_inr': 7849111,
 'closing_balance_inr': 2868895,
 'bounced_instruments': 0,
 'avg_daily_balance_inr': 2294870}

### FX Rates

In [ ]:
def get_fx_rate(from_curr: str) -> dict:
    """Retrieve FX rate for curreny conversion."""

    from_to = from_curr + "_INR"

    conn = sqlite3.connect("wealthpilot.db")
    conn.row_factory = sqlite3.Row

    row = conn.execute(
        """
        SELECT *
        FROM ref_rate_table
        WHERE rate = ?
        """,
        (from_to,)
    ).fetchone()

    conn.close()
    if row is None:
        return {"error": "FX rate not found"}
    return {f"FX rate for {from_curr} to INR": row['value']}

In [51]:
get_fx_rate('USD')

{'FX rate for USD to INR': 83.5}

### Risk Grade

In [76]:
_GRADES = [
("A1", 2.00, 750),
("A2", 1.75, 700),
("B1", 1.50, 650),
("B2", 1.35, 600),
("C1", 1.25, 550),
("C2", 1.15, 500),
("D1", 1.00, 450),
("D2", 0.00, 0)
]
_AUTO_GRADES = ("A1", "A2", "B1", "B2", "C1")

def compute_risk_grade(dscr: float | None, bureau_score: int | None) -> str | None:
    if dscr is None or bureau_score is None:
        return None
    return {'internal_grade': next(g for g, d, b, *_ in _GRADES if dscr >= d and bureau_score >= b)}

compute_risk_grade(1.93, 374)

{'internal_grade': 'D2'}

## Reliability wrappers: retry, circuit breaker, logging

- **Retry with backoff** — fail, wait a bit longer each time, try again.
- **Circuit breaker** — 3 failures *in a row* for the same tool trips it; stop even trying.
- **Logging** — every `(tool name, args, result-or-error)` recorded for later inspection.


In [117]:
import time, functools, random

def retry_with_backoff(max_retries: int = 3, base_delay: float = 0.5):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(max_retries):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    delay = base_delay * (2 ** attempt) + random.uniform(0, 0.1)
                    time.sleep(delay)
            raise last_exc
        return wrapper
    return decorator

class CircuitBreaker:
    def __init__(self, reset_timeout: float = 10.0):
        self.reset_timeout = reset_timeout
        self.state = "closed"          # closed -> open -> half_open -> closed
        self.opened_at = None

    def call(self, fn, fn_with_retry, *args, **kwargs):

        if self.state == "open":
            if self.reset_timeout - (time.monotonic() - self.opened_at) >= 0:
                time.sleep(self.reset_timeout - (time.monotonic() - self.opened_at))  # wait for the reset timeout to expire
            self.state = "half_open"
        try:
            if self.state == "half_open":
                result = fn(*args, **kwargs)  # single try
            else:
                result = fn_with_retry(*args, **kwargs)  # multiple retries with back-off
        except Exception:
            print("Tried max retries. Tool Calling Error. Circuit is open")
            self.state = "open"
            self.opened_at = time.monotonic()
            raise
        else:
            self.state = "closed"
            return result

TOOL_CALL_LOG: list[dict] = []

def make_robust_tool(fn, name: str):
    """Wraps fn with retry, circuit breaker and call logging"""

    breaker = CircuitBreaker(reset_timeout=10.0)
    fn_with_retry = retry_with_backoff(max_retries=3)(fn)

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        entry = {"tool": name, "args": args, "kwargs": kwargs, "ts": time.time()}
        # the first attempt tries max_retries attempts, and make the circuit open if all fail
        # the second attempt tries once after the reset time period
        for attempt in range(2): 
            try:
                result = breaker.call(fn, fn_with_retry, *args, **kwargs)
                entry["result"] = result
                entry["error"] = None
                TOOL_CALL_LOG.append(entry)
                return result
            except Exception as e:
                if attempt ==1:
                    entry["result"] = None
                    entry["error"] = str(e)
                    TOOL_CALL_LOG.append(entry)
                    raise
    return wrapper

print('Reliability wrappers defined')

Reliability wrappers defined


## Wire the tools into LangChain (LiteLLM underneath)


In [118]:
import json
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage

try:
    from langchain_litellm import ChatLiteLLM
    llm = ChatLiteLLM(model=LAB_MODEL, max_retries=5)
except Exception as e:
    print(f"ChatLiteLLM unavailable ({e}); the tool-calling loop below will use its offline fallback.")
    llm = None

robust_dscr = make_robust_tool(calculate_dscr, "calculate_dscr")
robust_sector_dscr = make_robust_tool(get_sector_dscr, "get_sector_dscr")
robust_credit_profile = make_robust_tool(get_credit_profile, "get_credit_profile")
robust_bank_statement = make_robust_tool(get_bank_statement, "get_bank_statement")
robust_fx_rate = make_robust_tool(get_fx_rate, "get_fx_rate")
robust_risk_grade = make_robust_tool(compute_risk_grade, "compute_risk_grade")

@tool
def dscr(applicant_id: str) -> dict:
    """Calculate DSCR (Debt Service Coverage Ratio) for the given applicant ID"""
    return robust_dscr(applicant_id)

@tool
def sector_dscr(sector_name: str) -> dict:
    """Get the sector DSCR (Debt Service Coverage Ratio) for the sector that the applicant's enterprise belongs to"""
    return robust_sector_dscr(sector_name)

@tool
def credit_profile(applicant_id: str) -> dict:
    """Get credit profile for the given applicant ID"""
    return robust_credit_profile(applicant_id)

@tool
def bank_statement(applicant_id: str) -> dict:
    """Get bank statement for the given applicant ID"""
    return robust_bank_statement(applicant_id)

@tool
def fx_rate(from_curr: str) -> dict:
    """Get fx rate for the given foreign currency -> INR"""
    return robust_fx_rate(from_curr)

@tool
def assess_risk(dscr: float, bureau_score: float) -> dict:
    """Compute the risk grade for an applicant"""
    return robust_risk_grade(dscr, bureau_score)

TOOLS = [dscr, sector_dscr, fx_rate, credit_profile, bank_statement, assess_risk]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}
print("LangChain tools declared:", list(TOOLS_BY_NAME))


LangChain tools declared: ['dscr', 'sector_dscr', 'fx_rate', 'credit_profile', 'bank_statement', 'assess_risk']


## The tool-call loop

In [119]:
SYSTEM_PROMPT = (
    "You are a risk-analysis agent. Use the available tools to assess the risk profile of the loan applicant. "
    "1. Calculate the DSCR for the given applicant"
    "2. Compare the calculated DSCR with the sector DSCR. Set meets_dscr_floor: True if dscr > sector_dscr_floor."
    "3. Get the bureau score from the applicant's credit profile"
    "4. Compute the risk grade of the applicant"
    "Then, reply your response in JSON format having information such as applicant_id, dscr, sector_dscr_floor, meets_dscr_floor"
    "bureau_score, and internal_grade"
)

def _fallback_risk_analysis_agent(intake_record: dict) -> str:
    """Deterministic, no-API demo path: calls the same real, wrapped tools in a fixed order, 
    so the infrastructure is provable without a live key."""
    TOOL_CALL_LOG.clear()

    underwriting = {}
    underwriting['applicant_id'] = intake_record['applicant_id']
    underwriting.update(robust_dscr(underwriting['applicant_id']))
    underwriting.update(robust_sector_dscr(underwriting['sector']))
    underwriting['meets_dscr_floor'] = underwriting['dscr'] >= underwriting['sector_dscr_floor']
    underwriting.update(robust_credit_profile(underwriting['applicant_id']))
    underwriting.update(robust_risk_grade(underwriting['dscr'], underwriting['bureau_score']))
    return underwriting

def run_risk_analysis_agent(raw_intake: str, max_iterations: int = 5) -> str:
    """Run the tool-calling loop until the model stops requesting tools or max_iterations is hit."""
    
    intake_record = json.loads(raw_intake)

    if llm is None or not _HAS_KEY:
        return _fallback_risk_analysis_agent(intake_record)

    model_with_tools = llm.bind_tools(TOOLS)
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=raw_intake)]

    for _ in range(max_iterations):
        ai_msg = model_with_tools.invoke(messages)
        messages.append(ai_msg)

        if not getattr(ai_msg, "tool_calls", None):
            return ai_msg.content

        for tc in ai_msg.tool_calls:
            tool_fn = TOOLS_BY_NAME[tc["name"]]
            try:
                result = tool_fn.invoke(tc["args"])
            except Exception as e:
                result = f"ERROR: {e}"
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return "Stopped: reached max_iterations without a final answer."

print("run_risk_analysis_agent defined.")


run_risk_analysis_agent defined.


In [120]:
TOOL_CALL_LOG.clear()
final_answer = run_risk_analysis_agent('{"applicant_id" : "ASH-A-00000"}')

In [122]:
print(final_answer)

```json
{
  "applicant_id": "ASH-A-00000",
  "dscr": 1.93,
  "sector_dscr_floor": 1.35,
  "meets_dscr_floor": true,
  "bureau_score": 374,
  "internal_grade": "D2"
}
```


In [123]:
TOOL_CALL_LOG

[{'tool': 'calculate_dscr',
  'args': ('ASH-A-00000',),
  'kwargs': {},
  'ts': 1792872613.6743972,
  'result': {'dscr': 1.93, 'sector': 'Light Engineering'},
  'error': None},
 {'tool': 'get_credit_profile',
  'args': ('ASH-A-00000',),
  'kwargs': {},
  'ts': 1792872613.6797674,
  'result': {'bureau_score': 374},
  'error': None},
 {'tool': 'get_sector_dscr',
  'args': ('Light Engineering',),
  'kwargs': {},
  'ts': 1792872619.3418455,
  'result': {'sector': 'Light Engineering', 'sector_dscr_floor': 1.35},
  'error': None},
 {'tool': 'compute_risk_grade',
  'args': (1.93, 374.0),
  'kwargs': {},
  'ts': 1792872625.1933572,
  'result': {'internal_grade': 'D2'},
  'error': None}]